<a href="https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## My Lane: Refresh / Content Opportunity Scoring

I'm choosing the **Refresh / Content Opportunity Scoring** core lane. In Weeks 1-2
I already built a hand-written rule and a decision tree that both attempt to flag
pages worth reviewing for refresh, using signals like staleness (days_since_last_update),
visibility (impressions_90d), and position. The starter data showed a real, non-trivial
"declining" rate (54.2% of pages), which means there's a genuine, sizeable problem here —
not a rare-event needle-in-a-haystack. This lane lets me build directly on that early
work instead of starting from scratch, and it maps cleanly onto something an SEO team
does every week: deciding which pages to refresh first.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## The Question

**Search question:** Which pages, out of a client's full content library, should be
prioritized for a content refresh this month, based on signs that they are stale and
declining in search performance?

**Unit of analysis:** A single page/URL (not a client, not a day) — each row in the
dataset is one page.

**Output:** A ranked list of pages (a "refresh queue") with a score, and a reason code
explaining why each page was flagged (e.g., "stale + high impressions + declining trend").

**The action:** A content editor or SEO strategist would take the top N pages from this
queue and schedule them for content refresh work (updating stats, rewriting sections,
adding new information) this sprint/month, instead of guessing which pages "feel" outdated.

**Cost of a wrong recommendation:**
- **False positive** (flagging a page that didn't need refreshing): wastes a writer's
  time and editorial budget on a page that wasn't actually declining — a few hours lost.
- **False negative** (missing a page that WAS declining and needed refresh): the page
  keeps losing rankings/traffic silently until someone notices manually, potentially
  costing weeks of lost organic visibility and revenue for that page.
- Because false negatives compound silently over time while false positives are just
  wasted effort, this lane should be tuned to lean toward catching more true decliners,
  even at some cost to precision.

**Why data/ML helps here:** With a large content library, no human can manually track
staleness + visibility + trend direction across thousands of pages every week. A
learned model can combine multiple weak signals (position, impressions, age, word count)
into a single ranked list far faster and more consistently than manual review — this
isn't a case where a simple SQL filter would do, because "declining" depends on the
interaction of several signals, not a single threshold.n All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ijazkhan0351-bot/Ijazweek1-ml-assignment"
REPO_DIR = "Ijazweek1-ml-assignment"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check repo root"
print("Data found, ready to go.")

Working dir: /content/Ijazweek1-ml-assignment
Data found, ready to go.


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Real number 1: how big is the "declining" problem?
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
declining_rate = df["is_declining_label"].mean()
print(f"1) Declining rate across all pages: {declining_rate:.1%} ({df['is_declining_label'].sum()} of {len(df)} pages)")

# Real number 2: how many pages are both stale AND still getting traffic (i.e. worth prioritizing)?
stale_and_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"2) Pages that are stale (180+ days) AND still visible (500+ impressions): {stale_and_visible} pages")

# Real number 3: how good was even a simple hand-written rule at this task?
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

hand_rule_p50 = precision_at_k(df["hand_rule_score"], df["is_declining_label"], 50)
print(f"3) A simple hand-written rule already hits Precision@50: {hand_rule_p50:.3f}")
print("This shows a real, usable signal exists in this data.")


1) Declining rate across all pages: 54.2% (16262 of 30000 pages)
2) Pages that are stale (180+ days) AND still visible (500+ impressions): 17 pages
3) A simple hand-written rule already hits Precision@50: 0.680
This shows a real, usable signal exists in this data.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## Careful Words

**What I can claim:** Within this anonymized sample, there is an observable,
directional relationship between staleness + visibility signals and whether a page
is declining. A simple rule already beats random guessing by a wide margin, which
suggests the signal is learnable.

**What I cannot claim:**
- I cannot claim this model would generalize to a different client's content, industry,
  or time period without validation on held-out, unseen pages.
- I cannot claim causation — a page being "stale" doesn't necessarily *cause* decline;
  both could be driven by a third factor (e.g., a competitor outranking it, an algorithm
  update, seasonality).
- I am not claiming to have "solved" refresh prioritization — this is an early,
  in-sample exploration, not a production-validated system.
- Any real deployment would need client-holdout validation (as the reference pipeline
  does), not the in-sample scoring shown here.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.